In [0]:
fetch_date=dbutils.widgets.get("fetch_date")
target_table=dbutils.widgets.get("target_table")
ac_insurance_table=dbutils.widgets.get("ac_insurance_table")
collector_table=dbutils.widgets.get("collector_table")
date_table=dbutils.widgets.get("date_table")

In [0]:
display(
spark.sql(f"""  
-- Step 1: Update Insurance ID from Insurance table
MERGE INTO {target_table} AS target
USING (
    SELECT
        Invoice_Number,
        Insurance_ID
    FROM (
        SELECT
            ma.Invoice_Number,
            ins.InsPolicy AS Insurance_ID,
            ROW_NUMBER() OVER (
                PARTITION BY ma.Invoice_Number
                ORDER BY ins.InsPolicy DESC
            ) AS rn
        FROM {target_table} ma
        LEFT JOIN {ac_insurance_table} ins
            ON ma.Invoice_Number = ins.AcctNbr
        WHERE ma.reporting_date = CAST('{fetch_date}' AS DATE)
    )
    WHERE rn = 1
) AS source
ON target.Invoice_Number = source.Invoice_Number
AND target.reporting_date = CAST('{fetch_date}' AS DATE)
WHEN MATCHED THEN
UPDATE SET target.Insurance_ID = source.Insurance_ID;
""")
)

In [0]:
display(
spark.sql(f"""  
-- Step 2: Clear NULL Insurance ID values
UPDATE {target_table}
SET Insurance_ID = ' '
WHERE Insurance_ID IS NULL
AND reporting_date = CAST('{fetch_date}' AS DATE)
""")
)

In [0]:
display(
spark.sql(f"""  
-- Step 3: Update Supervisor based on Collector and Quarter
-- CTE to calculate Quarter Number
WITH quarter_calc AS (
    SELECT 
        CASE QuarterInYear
            WHEN 1 THEN CONCAT(CalendarYear, '01')
            WHEN 2 THEN CONCAT(CalendarYear, '02')
            WHEN 3 THEN CONCAT(CalendarYear, '03')
            WHEN 4 THEN CONCAT(CalendarYear, '04')
        END AS QuarterNbr
    FROM {date_table}
    WHERE CalendarDate = CAST('{fetch_date}' AS DATE)
),

collector_supervisor AS (
    SELECT 
        clt.Collector,
        clt.Supervisor
    FROM {collector_table} clt
    INNER JOIN quarter_calc q
        ON clt.Quarter_Number = q.QuarterNbr
)
-- Merge Supervisor data into Master Aging
MERGE INTO {target_table} AS target
USING (
    SELECT DISTINCT
        ma.Collector,
        cs.Supervisor
    FROM {target_table} ma
    LEFT JOIN collector_supervisor cs
        ON ma.Collector = cs.Collector
    WHERE ma.reporting_date = CAST('{fetch_date}' AS DATE)
) AS source
ON target.Collector = source.Collector
AND target.reporting_date = CAST('{fetch_date}' AS DATE)

WHEN MATCHED THEN
    UPDATE SET target.Supervisor = source.Supervisor;
""")
)